# Getting started: accessing a SpatialData object

A Cytos cell screening run on the AVITI24 produces a **SpatialData object (SDO)**: a `.zarr.zip` archive containing cell-paint images, segmentation masks, per-molecule (polony) points, and per-cell measurement **tables**.

This tutorial covers the first steps:

1. Open a public SDO with `Loader` (stream from S3 by default, or use a local file).
2. Load its tables in Python.
3. Inspect the table structure.

No analysis is included. For field-level reference, see [`spatialdata_object_overview.md`](../docs/spatialdata_object_overview.md) and [`anndata_object_overview.md`](../docs/anndata_object_overview.md).


## 1. Access the data

Dataset: **Cytos-3Prime_HeLa-Tunicamycin** (~87.5 GB) — HeLa cells treated with tunicamycin, with 3′ RNA and imaging. No AWS credentials are required.

### Default: stream from public S3

The next cell opens the public `spatialdata/` prefix with `storage_options={"anon": True}`.
`Loader` reads tables and metadata remotely; image pyramids and molecule points stay on S3.

### Optional: local download

Download the archive when you need image pyramids, segmentation masks, or other pixel-level work.
Run the commands below from the **`tutorials/`** directory so the file is written to
`tutorials/data/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip`, then uncomment the local-override lines
in the next cell.

**Option A: `curl`**

```bash
mkdir -p data
curl -C - -L -o data/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip \
  "https://element-public-data.s3.us-west-2.amazonaws.com/Cytos/elembio-multiomics/Cytos-3Prime_HeLa-Tunicamycin/cb6f3a2ebb6c43acac4ad08b2b2a7779/spatialdata/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip"
```

If the download stops, run the same command again to resume. On Windows PowerShell, use `curl.exe`.

**Option B: AWS CLI**

Requires the [AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/getting-started-install.html):

```bash
mkdir -p data
aws s3 cp \
  s3://element-public-data/Cytos/elembio-multiomics/Cytos-3Prime_HeLa-Tunicamycin/cb6f3a2ebb6c43acac4ad08b2b2a7779/spatialdata/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip \
  data/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip \
  --no-sign-request
```

Do not unzip the archive; `Loader` reads `.zarr.zip` directly.


## 2. Load it

`Loader` opens the SDO and exposes per-modality tables. Install the analysis package first; see the
[installation instructions](../README.md#installation).


In [1]:
import pandas as pd
import s3fs
from IPython.display import Markdown, display

from elembio_spatialdata_analysis.load import Loader

# ---- default: stream from the public spatialdata/ prefix ----
SPATIALDATA_PREFIX = (
    "s3://element-public-data/Cytos/elembio-multiomics/"
    "Cytos-3Prime_HeLa-Tunicamycin/cb6f3a2ebb6c43acac4ad08b2b2a7779/spatialdata"
)
STORAGE_OPTIONS = {"anon": True}

# Optional local override (after downloading into tutorials/data/):
# SDO_PATH = "data/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip"
# STORAGE_OPTIONS = {}

fs = s3fs.S3FileSystem(**STORAGE_OPTIONS)
prefix_key = SPATIALDATA_PREFIX.removeprefix("s3://").rstrip("/")


def human_size(n: int) -> str:
    if n >= 1e9:
        return f"{n / 1e9:.2f} GB"
    if n >= 1e6:
        return f"{n / 1e6:.1f} MB"
    return f"{n / 1e3:.1f} kB"


def list_objects(key: str) -> pd.DataFrame:
    """Real objects directly under an S3 prefix, dropping directory markers."""
    columns = ["name", "size", "size_bytes", "uri"]
    try:
        entries = fs.ls(key, detail=True)
    except FileNotFoundError:
        return pd.DataFrame(columns=columns)
    rows = [
        {
            "name": entry["name"].rsplit("/", 1)[-1],
            "size": human_size(entry["size"]),
            "size_bytes": entry["size"],
            "uri": "s3://" + entry["name"],
        }
        for entry in entries
        if entry["type"] == "file" and entry["name"].rsplit("/", 1)[-1]
    ]
    return pd.DataFrame(rows, columns=columns).sort_values("size_bytes", ascending=False).reset_index(drop=True)


if "SDO_PATH" not in globals():
    prefix_objects = list_objects(prefix_key)
    sdo_candidates = prefix_objects.loc[prefix_objects["name"].str.endswith(".zarr.zip")]
    if len(sdo_candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one *.zarr.zip under {SPATIALDATA_PREFIX}, found {len(sdo_candidates)}."
        )
    SDO_PATH = sdo_candidates["uri"].iloc[0]
    display(Markdown("### Objects in the public prefix"))
    display(prefix_objects[["name", "size"]])
    print("SDO:    ", SDO_PATH)
    print("access: anonymous (public bucket)")

loader = Loader(SDO_PATH, storage_options=STORAGE_OPTIONS)
print(loader.summary())


### Objects in the public prefix

,name,size
0,Cytos-3Prime_HeLa-Tunicamycin.zarr.zip,87.49 GB
1,points-zarr.zip,4.02 GB
2,cell-shapes-zarr.zip,53.9 MB
3,nuclear-shapes-zarr.zip,19.0 MB
4,cyto.viz,14.3 MB
5,Cytos-3Prime_HeLa-Tunicamycin.zarr-index.bin,874.1 kB
6,Cytos-3Prime_HeLa-Tunicamycin.points-zarr-inde...,0.3 kB
7,Cytos-3Prime_HeLa-Tunicamycin.nuclear-shapes-z...,0.1 kB
8,Cytos-3Prime_HeLa-Tunicamycin.cell-shapes-zarr...,0.1 kB


SDO:     s3://element-public-data/Cytos/elembio-multiomics/Cytos-3Prime_HeLa-Tunicamycin/cb6f3a2ebb6c43acac4ad08b2b2a7779/spatialdata/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip
access: anonymous (public bucket)


--- Cytos-3Prime_HeLa-Tunicamycin ---
  run_name:       Cytos-3Prime_HeLa-Tunicamycin
  source:         s3://element-public-data/Cytos/elembio-multiomics/Cytos-3Prime_HeLa-Tunicamycin/cb6f3a2ebb6c43acac4ad08b2b2a7779/spatialdata/Cytos-3Prime_HeLa-Tunicamycin.zarr.zip
  tables:         ['ThreePrimeUntargeted', 'imaging']
  batches (7):   B01 (ThreePrimeUntargeted), B02 (ThreePrimeUntargeted), B03 (ThreePrimeUntargeted), B04 (ThreePrimeUntargeted), B05 (ThreePrimeUntargeted), CP01 (PreAmp), P01 (PreAmp)
  wells (12):     A1, A2, B1, B2, C1, C2, D1, D2, ... (12 total)
  tiles:          144
  points:         60 elements (well mode)
  build:          1.0.0rc15


`summary()` prints a quick overview of the run: its name, the tables it contains, the sequencing/imaging batches, the wells, and how many tiles and point elements there are.

## 3. What are the tables?

The measurements live in **tables**. Each table is an [AnnData](https://anndata.readthedocs.io/) object, a **cell × feature** matrix:

- **rows** are individual segmented cells
- **columns** are measured features (genes, proteins, or morphology measurements)

Which tables are present depends on what the run measured. This dataset contains two:

| Table | What it measures |
|-------|------------------|
| `ThreePrimeUntargeted` | RNA counts per cell |
| `imaging` | cell morphology and protein features derived from the images|

Other runs may contain additional tables, such as the OPS identity and control-panel tables in the CRISPR-screen case study (named after the panel). The loader discovers whatever is present.

Every modality table describes the **same cells in the same order**, so a given row is the same physical cell across all tables. The shared key is `cell_id_global`.

In [2]:
# Which tables and modalities does this run contain?
print('Tables in this run:  ', sorted(loader.available_tables))
print('Modalities measured: ', sorted(loader.available_target_types))

Tables in this run:   ['ThreePrimeUntargeted', 'imaging']
Modalities measured:  ['ThreePrimeUntargeted', 'imaging']


In [3]:
# Load one modality table as an AnnData object.
# lazy=False materializes the measurement matrix into memory.
table_name = 'ThreePrimeUntargeted'
adata = loader.load_tables(table=table_name, lazy=False)

print(f'Loaded {table_name!r}')
adata

Loaded 'ThreePrimeUntargeted'


AnnData object with n_obs × n_vars = 150732 × 54949
    obs: 'Cell', 'Well', 'WellLabel', 'tile_id', 'X_tile', 'Y_tile', 'cell_id_local', 'cell_id_global', 'region_key', 'well_id'
    var: 'seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'gene_id', 'gene_type', 'level', 'hgnc_id', 'havana_gene', 'mane_select_transcript_id', 'mane_select_transcript_name', 'batch_name', 'target_type'
    uns: 'metadata_attrs', 'spatial_artifact_params'
    obsm: 'total_counts', 'data_missing', 'spatial_artifacts', 'spatial', 'cell_geometry'
    layers: 'nuclear'

The summary above lists the parts of an AnnData object. The three you will use most:

- **`.X`**: the measurement matrix (cells × features).
- **`.obs`**: a table of per-cell annotations, one row per cell (well, `cell_id_global`, ...).
- **`.var`**: a table of per-feature annotations, one row per gene / protein / feature.

(`.obsm`, `.layers`, and `.uns` hold extra per-cell matrices, alternative count layers, and unstructured metadata; the reference docs cover these in full.)

Let’s look at each.

In [4]:
# Shape is (cells, features). Per-cell annotations live in .obs:
print('shape (cells, features):', adata.shape)
adata.obs.head()

shape (cells, features): (150732, 54949)


,Cell,Well,WellLabel,tile_id,X_tile,Y_tile,cell_id_local,cell_id_global,region_key,well_id
1,1,A1,Hela,L2R01C01S1,4223.656738,66.987862,1,1,A1-labels-Cell,A1
2,2,A1,Hela,L2R01C01S1,4672.289551,38.415596,2,2,A1-labels-Cell,A1
3,3,A1,Hela,L2R01C01S1,977.739136,71.378769,3,3,A1-labels-Cell,A1
4,4,A1,Hela,L2R01C01S1,1616.999023,56.193249,4,4,A1-labels-Cell,A1
5,5,A1,Hela,L2R01C01S1,1677.071655,52.519173,5,5,A1-labels-Cell,A1


In [5]:
# Per-feature annotations live in .var (one row per gene / protein / feature):
adata.var.head()

,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_type,level,hgnc_id,havana_gene,mane_select_transcript_id,mane_select_transcript_name,batch_name,target_type
A1BG,chr19,HAVANA,gene,58345178,58353492,.,-,.,ENSG00000121410.12,protein_coding,1,HGNC:5,OTTHUMG00000183507.3,ENST00000263100.8,A1BG-201,B01,ThreePrimeUntargeted
A1BG-AS1,chr19,HAVANA,gene,58347718,58355455,.,+,.,ENSG00000268895.6,lncRNA,2,HGNC:37133,OTTHUMG00000183508.2,None,None,B01,ThreePrimeUntargeted
A1CF,chr10,HAVANA,gene,50799409,50885675,.,-,.,ENSG00000148584.16,protein_coding,2,HGNC:24086,OTTHUMG00000018240.6,ENST00000373997.8,A1CF-203,B01,ThreePrimeUntargeted
A2M,chr12,HAVANA,gene,9067664,9116229,.,-,.,ENSG00000175899.15,protein_coding,2,HGNC:7,OTTHUMG00000150267.7,ENST00000318602.12,A2M-201,B01,ThreePrimeUntargeted
A2M-AS1,chr12,HAVANA,gene,9065163,9068689,.,+,.,ENSG00000245105.5,lncRNA,2,HGNC:27057,OTTHUMG00000168289.4,None,None,B01,ThreePrimeUntargeted


In [6]:
# cell_id_global is the key that lines cells up across every table in the run.
adata.obs['cell_id_global'].head()

1    1
2    2
3    3
4    4
5    5
Name: cell_id_global, dtype: uint32

## Optional: extract just the tables

Most of the `.zarr.zip` size is images, labels, shapes, and points. After opening a `Loader`
(from S3 or a local file), export the per-cell tables to local `.h5ad` files:

```python
paths = loader.export_tables("data/tables", fmt="h5ad")
```

Streaming from S3 and exporting tables avoids downloading image pyramids.

Each export includes:

- One `{run}__{table}.h5ad` per modality (plus a small `metadata` file by default)
- Provenance in `uns['export_provenance']`
- Spatial coordinates in the table (not image pyramids or polony points)

Open an exported table with AnnData:

```python
import anndata as ad
adata = ad.read_h5ad("data/tables/Cytos-3Prime_HeLa-Tunicamycin__ThreePrimeUntargeted.h5ad")
```

The [case studies](case-studies/) also use `Loader` on the public archive for run metadata and
multi-modality analysis. Use exported `.h5ad` files when you only need the matrices.


## Next steps

You now have an SDO loaded and know how to pull a cell × feature table out of it. From here:

- Load another modality with `loader.load_tables(table=...)`. The loader returns every table with the same cells in the same row order, so modalities are already aligned. You can use them together without joining (`cell_id_global` is the shared identifier if you ever need to align tables yourself).
- Optionally [`export_tables`](#optional-extract-just-the-tables) to keep only the matrices locally.
- Try a case study: [`ops-crispr-tgfbr2-tnfa.ipynb`](case-studies/ops-crispr-tgfbr2-tnfa.ipynb) or [`hela-tunicamycin-upr.ipynb`](case-studies/hela-tunicamycin-upr.ipynb).
- For the field-by-field reference of every table, see [`anndata_object_overview.md`](../docs/anndata_object_overview.md). For the SDO container as a whole (images, labels, shapes, points), see [`spatialdata_object_overview.md`](../docs/spatialdata_object_overview.md).
